In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-10.3.9

In [2]:
import yaml
import os
import csv
from ultralytics import YOLO
import random
import torch
import gc
from glob import glob
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from ultralytics.utils.metrics import ConfusionMatrix

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!cp -r /kaggle/input/rasoul-b-obb/dataset /kaggle/working/

In [4]:
# Define base directories
BASE_DIR = "/kaggle/working/yolo_training_results"
PREDICTIONS_DIR = "/kaggle/working/yolo_validation_predictions"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

In [5]:
# Generating yaml file
# Define YOLO dataset structure
data = {
    'train': '/kaggle/working/dataset/images/train',
    'val': '/kaggle/working/dataset/images/val',
    'test': '/kaggle/working/dataset/images/test',
    'nc': 1,
    'names': ['ICH'],
    'obb': True
}

# Save YAML file
yaml_path = "/kaggle/working/dataset/dataset.yaml"

with open(yaml_path, 'w') as file:
    yaml.dump(data, file, default_flow_style=False)

print(f"YAML file saved at: {yaml_path}")

YAML file saved at: /kaggle/working/dataset/dataset.yaml


In [6]:
# Set the seed value
SEED = 42  

# Python’s built-in random module
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Ensure deterministic behavior in PyTorch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set PYTHONHASHSEED to ensure consistent behavior in hash-based operations
os.environ["PYTHONHASHSEED"] = str(SEED)

In [7]:
# Model configurations
models = [
    "yolov8n-obb"
]

# Training parameters
EPOCHS = 11
BATCH_SIZE = 4

# Store results for visualization
results_dict = {}

In [8]:
# Loop through each model
for model_name in models:

    print(f"\033[1m\n🚀 Training started for {model_name}...\033[0m\n")
    torch.cuda.empty_cache()
    gc.collect()

    model = YOLO(f"{model_name}.pt")

    # Train the model
    model.train(
        data=yaml_path,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        device=[0,1],
        augment=False,
        seed=SEED,
        project=BASE_DIR,
        name=model_name,
        save=True,
        cache=True,
        val=True,
        save_txt=True,
        save_conf=True,

        # Disable augmentations
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        translate=0.0, scale=0.0, shear=0.0, degrees=0.0, perspective=0.0,
        fliplr=0.0, flipud=0.0,
        mosaic=0.0, mixup=0.0, copy_paste=0.0, auto_augment=None,
        erasing=0.0
    )

    # Updated result_dir
    result_dir = os.path.join(BASE_DIR, model_name)

    # Store CSV results
    results_dict[model_name] = os.path.join(result_dir, "results.csv")

    print(f"\033[1m✅ Training completed for {model_name}. Results saved in {result_dir}\033[0m\n")

    del model
    torch.cuda.empty_cache()
    gc.collect()


🚀 Training started for yolov8n-obb...

Ultralytics 8.3.203 🚀 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=None, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=11, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=yolov8n-obb, n

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1990        822      0.756      0.488       0.61      0.371
Speed: 0.5ms preprocess, 7.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /kaggle/working/yolo_training_results/yolov8n-obb
✅ Training completed for yolov8n-obb. Results saved in /kaggle/working/yolo_training_results/yolov8n-obb

